# A5 H3 — 지정 조건 관측 파일럿

기존 company_size 호출에서 조건 근거를 먼저 출력하는 후보와 대조군을 비교합니다.
각 군은 dev 200건 + 이미 감사한 진단 5건이며 한 회차 410응답입니다.
기본/SME 단계는 보관 응답으로 고정한 **혼합 CPU 재생**입니다. 전체 GPU 점수나 제출물이 아닙니다.

1. A100 GPU 런타임을 선택하고 HF_TOKEN 보안 비밀 접근을 허용합니다.
2. 기존 Drive의 `MyDrive/a5/train_unlabeled.jsonl`을 사용합니다.
3. 첫 실행은 `EPISODE = 1`, 위에서부터 모두 실행합니다.
4. 결과 ZIP을 보관합니다. 시간 검사 통과 시 런타임을 삭제하고 새 A100 런타임에서 같은 노트북을 `EPISODE = 2`로 실행합니다.
5. 회차 2는 순서를 뒤집고 같은 군의 반복 차이도 저장합니다. 기존 결과 폴더는 덮어쓰지 않습니다.

과거 A100 기준 추론/준비 약 9~12분 + 모델 적재 약 7.5분 예상이며 설치·다운로드는 별도입니다.
H3 실제 속도는 미측정입니다. 기존 H2의 남은 무라벨 18회차를 재개하지 않습니다.


In [ ]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="a5-scope-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "173533093617e340f4d4d5858d39766726cf0488"  # 회차 기록을 포함한 H3 파일럿 코드

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)

## 코드 고정

In [ ]:
REPO = WORK / "repo"
run_logged("git-clone", ["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
run_logged("git-fetch", ["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO)
run_logged("git-checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if SOURCE_COMMIT != REPO_REF:
    raise ValueError("REPO_REF를 실행 안내의 40자리 SHA로 고정하세요.")
SUBMISSION = REPO  # 기존 고정 환경 설치 셀에서 requirements.txt 경로로만 사용
write_json(RESULTS / "source.json", {"commit": SOURCE_COMMIT, "requested_ref": REPO_REF})
assert (REPO / "experiments/a5_scope_pilot.py").is_file()

## Drive 입력과 회차 선택

기존 원본 파일을 그대로 사용합니다. 없으면 PC의 `open/train_unlabeled.jsonl`을
Drive의 `내 드라이브/a5/`에 업로드한 뒤 아래 셀부터 다시 실행하세요.
회차 2는 회차 1을 완주했고 H3 시간 계획 검사를 통과한 뒤 새 GPU 런타임에서 실행합니다.
실패한 회차를 다시 실행할 때는 `ATTEMPT`를 새로운 이름으로 바꾸면 기존 로그를 보존합니다.
두 회차 모두 같은 ATTEMPT를 사용합니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
EPISODE = 1  # 첫 런타임 1, 새 런타임 반복은 2
ATTEMPT = "a"  # 실패 후 새 시도는 b 등으로 변경. 기존 결과 보존.
assert EPISODE in (1, 2)
assert ATTEMPT.isalnum() and len(ATTEMPT) <= 16
UNLABELED_PATH = Path("/content/drive/MyDrive/a5/train_unlabeled.jsonl")
PILOT_ROOT = Path("/content/drive/MyDrive/a5/h3-scope-" + SOURCE_COMMIT[:12] + "-" + ATTEMPT)
FACTS_OUTPUT = PILOT_ROOT / ("episode-" + str(EPISODE))
if FACTS_OUTPUT.exists():
    raise FileExistsError("회차 결과가 이미 있습니다. 덮어쓰지 않습니다. EPISODE 또는 ATTEMPT를 확인하세요.")
if EPISODE == 2:
    previous = json.loads((PILOT_ROOT / "episode-1/summary.json").read_text(encoding="utf-8"))
    assert previous["complete"], "회차 1이 완료되지 않았습니다. 먼저 기존 로그를 확인하세요."
    assert previous["arms"]["h3"]["within_stage_planning_limit"], "회차 1이 시간 계획 상한을 초과했습니다. 반복 전에 결과를 검토하세요."
if not UNLABELED_PATH.is_file():
    write_json(RESULTS / "input-check.json", {"status": "missing", "path": str(UNLABELED_PATH)})
    raise FileNotFoundError("PC의 open/train_unlabeled.jsonl을 Drive의 내 드라이브/a5에 업로드한 뒤 이 셀을 다시 실행하세요.")
with UNLABELED_PATH.open("rb") as f:
    input_hash = hashlib.file_digest(f, "sha256").hexdigest()
assert input_hash == "f46449fb84f980a5ddf868d66f5daff2bcf0991135d9b81da5656dd607275698", "원본 입력 SHA256 불일치"
write_json(RESULTS / "input-check.json", {"status": "verified", "sha256": input_hash, "episode": EPISODE, "attempt": ATTEMPT})
print("고정 코드:", SOURCE_COMMIT, "이번 결과:", FACTS_OUTPUT)


## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.

In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")

## 고정 Python·추론 패키지 설치

기존 Colab 검증과 같은 Python 3.12.13·vLLM 0.26.0·CUDA 13.0을 사용합니다.

In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 같은 저장소의 requirements.txt를 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")

## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.

In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})

## 대조군·H3 실행과 혼합 재생

회차 1은 control→H3, 회차 2는 H3→control 순서입니다. 원응답과 문서 예산은 군별로 저장합니다.
새 관측 필드가 판정을 직접 덮어쓰지는 않습니다. 동일 H2 규칙에 새 사실을 넣어 효과를 측정합니다.
각 군의 `head-hybrid.csv`/`h2-hybrid.csv`, 24항목 지표와 `*-comparison.json`이 생성됩니다.
339.178초/200건은 다른 단계 비용이 동일하다는 조건의 계획 상한입니다. 서버 통과 보장이 아닙니다.
초과해도 진단 결과는 보존하며 회차 2 자동 진행을 막습니다. 실패 시 마지막 다운로드 셀을 실행하세요.


In [ ]:
inference_env = dict(os.environ)
inference_env.pop("HF_TOKEN", None)
inference_env.pop("HUGGING_FACE_HUB_TOKEN", None)
run_logged("a5-scope", [PYTHON, str(REPO / "experiments/a5_scope_pilot.py"),
    "--input", str(UNLABELED_PATH), "--output-dir", str(FACTS_OUTPUT),
    "--model-dir", MODEL_DIR, "--episode", str(EPISODE)], env=inference_env, cwd=REPO)
summary = json.loads((FACTS_OUTPUT / "summary.json").read_text(encoding="utf-8"))
print(json.dumps({name: {"macro_f1": arm["metrics"]["h2"]["macro_f1"], "v11": arm["metrics"]["h2"]["items"]["v11"],
                       "stage_seconds": arm["dev_stage_seconds"],
                       "within_limit": arm["within_stage_planning_limit"]}
                  for name, arm in summary["arms"].items()}, ensure_ascii=False, indent=2))
if EPISODE == 2:
    first = PILOT_ROOT / "episode-1"
    old_contract = json.loads((first / "contract.json").read_text(encoding="utf-8"))
    new_contract = json.loads((FACTS_OUTPUT / "contract.json").read_text(encoding="utf-8"))
    for key in old_contract.keys() - {"episode", "order"}:
        assert old_contract[key] == new_contract[key], "회차 계약 불일치: " + key
    assert json.loads((first / "environment.json").read_text())["environment"] == json.loads((FACTS_OUTPUT / "environment.json").read_text())["environment"], "GPU/환경 불일치"
    for arm in ("control", "h3"):
        for variant in ("head", "h2"):
            run_logged("repeat-" + arm + "-" + variant, [PYTHON, str(REPO / "tools/compare_runs.py"),
                "--before", str(first / arm / (variant + "-hybrid.csv")),
                "--after", str(FACTS_OUTPUT / arm / (variant + "-hybrid.csv")),
                "--items", "v11,v13", "--output-dir", str(FACTS_OUTPUT / ("repeat-" + arm + "-" + variant))], cwd=REPO)
print("시간 계획 통과. 회차 1이면 새 런타임에서 회차 2를 실행하세요." if summary["arms"]["h3"]["within_stage_planning_limit"] and EPISODE == 1 else "결과 ZIP을 보관해 검토하세요. 채택/서버 검증은 완료되지 않았습니다.")


## 결과 ZIP 다운로드 — 실패했어도 이 셀 실행

설치/실행 로그와 같은 ATTEMPT의 회차 1·2 원응답·조건 검사·혼합 재생을 묶습니다.
ZIP을 전달하면 조건 관측, 대상/공유 항목 변화, 반복 churn, 시간 예산을 대조할 수 있습니다.
이 노트북은 대회 제출 또는 모델 가중치 변경을 하지 않습니다.


In [ ]:
from google.colab import files

archive_path = WORK / ("a5-scope-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, "logs/" + path.relative_to(RESULTS).as_posix())
    if "PILOT_ROOT" in globals() and PILOT_ROOT.exists():
        for path in sorted(PILOT_ROOT.rglob("*")):
            if path.is_file() and not path.name.endswith(".partial"):
                archive.write(path, "pilot/" + path.relative_to(PILOT_ROOT).as_posix())
files.download(str(archive_path))
